[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C58_HardCase_LongTail_Course/01_imbalance/01_class_imbalance.ipynb)

# 01 · 类别不平衡的处理谱系（RFS / CB Loss / LDAM / EQLv2 / logit adjustment / 解耦）

目标：在**同一份合成长尾数据**上把整条方法谱系跑一遍，亲手看到每个方法把尾部类召回抬了多少、
把头部类压了多少——**而不是背方法名**。

本 notebook 你会亲手实现：
1. 幂律长尾分类任务 + 基线 softmax 分类器（以及"平衡数据 oracle"作为上界）
2. **Repeat Factor Sampling**：`r_c = max(1, √(t/f_c))` + 图内取最大 + 每 epoch 随机取整
3. **Class-Balanced Loss** 的有效样本数 `(1-β^n)/(1-β)`（含递推式验证）与 **LDAM** 的间隔
4. **负梯度淹没的复现**：sigmoid 多标签头上，稀有类的累积正负梯度比 → **EQL v1 与 EQLv2**
5. **Logit adjustment**：推理时减 `τ·log π_c`，零训练成本
6. **解耦训练**：`‖w_c‖ vs log n_c` 的相关性、**NCM 诊断**（证明表示是好的）、τ-normalize、cRT
7. **全方法对比表** + **方法选择决策树**的代码化

> 心智模型：**稀有类的问题往往不是"特征学不好"，而是"分类器的 logit 被海量负梯度压死了"。**
> 先诊断是表示问题还是分类器问题，再按成本递增选药。

## 1 · 合成长尾数据集与基线

合成规则（都写清楚，方便你改参数看规律）：
- `C=12` 个类，第 `r` 名的训练样本数 `n_r = 1500 · r^-2.0`，下限 10 → **头/尾比 150:1**
- 每类一个 24 维球面上的中心（模长 3.0），样本 = 中心 + 各向同性高斯噪声（σ=1）
- **表示层固定**：`h = tanh(x·R + b_R)`，`R` 是固定的随机投影。
  这样所有方法都在同一份表示上比较分类器——**正是本模块要隔离的变量**
- **测试集类别平衡**（每类 300 个），因为我们关心的是 macro 指标

In [ ]:
import numpy as np, json, math
rng = np.random.default_rng(58)
np.set_printoptions(precision=3, suppress=True)

C, D, H = 12, 24, 32
ALPHA, N_HEAD, N_FLOOR = 2.0, 1500, 10
ranks = np.arange(1, C + 1)
train_counts = np.maximum(np.round(N_HEAD * ranks ** (-ALPHA)), N_FLOOR).astype(int)

centers = rng.normal(size=(C, D))
centers /= np.linalg.norm(centers, axis=1, keepdims=True)
centers *= 3.0                                   # 类间距离 ≈ 3√2 ≈ 4.24, 噪声 σ=1

def sample(counts, gen):
    X = np.concatenate([centers[c] + gen.normal(size=(n, D)) for c, n in enumerate(counts)])
    y = np.concatenate([np.full(n, c) for c, n in enumerate(counts)])
    return X, y

Xtr, ytr = sample(train_counts, rng)
Xte, yte = sample(np.full(C, 300), np.random.default_rng(7))     # **平衡测试集**

R = rng.normal(size=(D, H)) / np.sqrt(D)
bR = rng.normal(size=H) * 0.1
def feat(X): return np.tanh(X @ R + bR)          # 固定表示层
Htr, Hte = feat(Xtr), feat(Xte)

HEAD, MID, TAIL = slice(0, 4), slice(4, 8), slice(8, 12)
print('每类训练样本数:', train_counts, ' 总计', train_counts.sum())
print(f'头/尾比 = {train_counts[0] / train_counts[-1]:.0f} : 1'
      f'   |  head={train_counts[HEAD].sum()}  mid={train_counts[MID].sum()}'
      f'  tail={train_counts[TAIL].sum()}')

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def train_softmax(Hm, y, sample_w=None, class_w=None, margin=None, prior_logit=None,
                  steps=1500, lr=0.5, wd=1e-4):
    '''全批梯度下降的线性 softmax 分类器。
       sample_w  : 每样本权重（用来模拟重采样 —— 重复 k 次 ≡ 权重 k）
       class_w   : 每类权重（重加权：CB Loss / 反频率）
       margin    : 每类间隔 Δ_c（LDAM：训练时 z_y ← z_y - Δ_y）
       prior_logit: 训练时的先验偏置 τ·log π（logit-adjusted loss）'''
    n = len(y)
    W = np.zeros((Hm.shape[1], C)); b = np.zeros(C)
    Y = np.zeros((n, C)); Y[np.arange(n), y] = 1.0
    w = np.ones(n) if sample_w is None else np.asarray(sample_w, float)
    if class_w is not None:
        w = w * np.asarray(class_w, float)[y]
    w = w / w.mean()
    for _ in range(steps):
        z = Hm @ W + b
        if margin is not None:  z = z - Y * margin           # 只对真值类扣间隔
        if prior_logit is not None: z = z + prior_logit
        G = (softmax(z) - Y) * w[:, None]
        W -= lr * (Hm.T @ G) / n + lr * wd * W
        b -= lr * G.mean(0)
    return W, b

def per_class_acc(W, b, Hm=None, y=None, adj=None):
    Hm = Hte if Hm is None else Hm
    y = yte if y is None else y
    z = Hm @ W + b
    if adj is not None: z = z + adj
    pred = z.argmax(1)
    return np.array([(pred[y == c] == c).mean() for c in range(C)])

RESULTS = {}
def report(name, per, store=True):
    if store: RESULTS[name] = per
    print(f'{name:<26s} head {per[HEAD].mean():.3f}  mid {per[MID].mean():.3f}'
          f'  tail {per[TAIL].mean():.3f}  **macro {per.mean():.3f}**')

W0, b0 = train_softmax(Htr, ytr)
per0 = per_class_acc(W0, b0)
report('① baseline (plain CE)', per0)
print('   逐类准确率:', np.round(per0, 3))

Xb, yb = sample(np.full(C, N_HEAD), np.random.default_rng(3))    # 上界: 每类都有 1500 个
Wb, bb = train_softmax(feat(Xb), yb)
report('⓪ oracle (平衡数据)', per_class_acc(Wb, bb))

assert per0[HEAD].mean() > per0[TAIL].mean() + 0.3, '基线必须呈现明显的头尾差'
assert RESULTS['⓪ oracle (平衡数据)'].mean() > per0.mean() + 0.15, 'oracle 应显著更高'
print('\n⚠️  基线的头尾差 %.3f —— 这不是"特征不够好", 后面第 6 节会证明表示其实没坏。'
      % (per0[HEAD].mean() - per0[TAIL].mean()))

## 2 · Repeat Factor Sampling（LVIS 的做法）

`r_c = max(1, √(t/f_c))` → `r_i = max_{c∈i} r_c` → 每 epoch 随机取整。

**注意 `f_c` 是图像频率（含该类的图像数 ÷ 总图像数），不是实例数占比**——
因为采样的单位是图像。本 notebook 里每个样本 = 一张只含一个目标的图，两者恰好相同；
练习 1 会让你实现真正的多标签版本（一张图含多个类）。

In [ ]:
f_img = train_counts / train_counts.sum()        # 图像频率 f_c

def repeat_factor_c(f_c, t):
    '''类别级 repeat factor: r_c = max(1, sqrt(t / f_c))'''
    return np.maximum(1.0, np.sqrt(t / np.asarray(f_c, float)))

print(f"{'阈值 t':>8s}  " + ' '.join(f'c{c:<4d}' for c in [0, 3, 7, 11]))
for t in [0.01, 0.05, 0.2]:
    r = repeat_factor_c(f_img, t)
    print(f'{t:>8.2f}  ' + ' '.join(f'{r[c]:<5.2f}' for c in [0, 3, 7, 11]))
print('\n对比 **完全拉平** 1/f_c 的重复倍数:', np.round(1 / f_img, 1)[[0, 3, 7, 11]])
print('→ √ 把尾部重复倍数从 %.0f× 压到 %.1f×, 这就是"温和过采样"的含义。'
      % ((1 / f_img)[-1], repeat_factor_c(f_img, 0.2)[-1]))

T_RFS = 0.2
r_c = repeat_factor_c(f_img, T_RFS)

def rfs_epoch(y, r_c, gen):
    '''每 epoch 随机取整: k_i = floor(r_i) + Bernoulli(r_i - floor(r_i))'''
    r = r_c[y]
    k = np.floor(r).astype(int) + (gen.random(len(r)) < (r - np.floor(r))).astype(int)
    return np.repeat(np.arange(len(y)), k)

gen = np.random.default_rng(5)
idx = rfs_epoch(ytr, r_c, gen)
print(f'\nepoch 长度: {len(ytr)} → {len(idx)}  ({len(idx)/len(ytr):.2f}×)'
      '   ← **消融时必须同步调 iteration 数, 否则重采样组偷偷多训了**')

# 期望重复次数 ≈ r_i（随机取整是无偏的）
k_exp = np.zeros(C)
for _ in range(30):
    ii = rfs_epoch(ytr, r_c, gen)
    k_exp += np.bincount(ytr[ii], minlength=C) / train_counts
k_exp /= 30
print('随机取整 30 个 epoch 的平均重复次数:', np.round(k_exp, 2))
assert np.allclose(k_exp, r_c, atol=0.06), '随机取整应当是无偏的'

sw = np.bincount(rfs_epoch(ytr, r_c, gen), minlength=len(ytr)).astype(float)
Wr, br = train_softmax(Htr, ytr, sample_w=sw)
report('② RFS (t=0.2)', per_class_acc(Wr, br))
assert RESULTS['② RFS (t=0.2)'][TAIL].mean() > per0[TAIL].mean() + 0.1, 'RFS 应显著抬高尾部'
print('\n✅ RFS 的四个设计点: **图像频率 / 开方 / 图内取最大 / 每 epoch 随机取整**。')
print('⚠️  它只增加"看到的次数", 不增加信息量 —— 10 张图重复 100 次 ≠ 1000 张图。')

## 3 · Class-Balanced Loss 的有效样本数，与 LDAM 的间隔

`E_n = (1-β^n)/(1-β)` 来自递推 `E_n = 1 + β·E_{n-1}, E_1 = 1`
（假设每个新样本有 β 的概率落进已被覆盖的区域）。
**β 是把"不加权"与"反频率"连成一条连续曲线的旋钮**，不是二选一。

In [ ]:
def effective_num(n, beta):
    n = np.asarray(n, float)
    if beta <= 0: return np.ones_like(n)
    return (1.0 - beta ** n) / (1.0 - beta)

# 递推式验证: E_n = 1 + beta * E_{n-1}
for beta in [0.9, 0.99, 0.999]:
    E = 1.0
    for k in range(2, 60):
        E = 1.0 + beta * E
    assert np.isclose(E, effective_num(59, beta)), f'递推与闭式应一致 (beta={beta})'
print('✅ 递推 E_n = 1 + β·E_{n-1} 与闭式 (1-β^n)/(1-β) 完全一致')

print(f"\n{'β':>8s}{'1/(1-β)':>10s}   " + ''.join(f'{f"E(n={n})":>11s}' for n in [10, 100, 1500]))
for beta in [0.0, 0.9, 0.99, 0.999, 0.9999]:
    E = effective_num([10, 100, 1500], beta)
    sat = f'{1/(1-beta):.0f}' if beta < 1 else '∞'
    print(f'{beta:>8.4f}{sat:>10s}   ' + ''.join(f'{e:>11.1f}' for e in E))
print('→ β=0: E_n≡1(等价不加权)   β→1: E_n→n(等价反频率)。**1/(1-β) ≈ "多少样本算饱和"**')
assert np.allclose(effective_num([10, 100], 0.0), [1, 1])
assert abs(effective_num([50], 0.999999)[0] - 50) < 0.01, 'β→1 时 E_n → n'

def cb_weights(counts, beta):
    w = 1.0 / effective_num(counts, beta)
    return w / w.mean()

print(f"\n{'方案':<22s}" + ''.join(f'{f"w(c{c})":>10s}' for c in [0, 3, 7, 11]))
for name, w in [('不加权', np.ones(C)),
                ('CB β=0.99', cb_weights(train_counts, 0.99)),
                ('CB β=0.999', cb_weights(train_counts, 0.999)),
                ('CB β=0.9999', cb_weights(train_counts, 0.9999)),
                ('反频率 1/n', (1 / train_counts) / (1 / train_counts).mean())]:
    print(f'{name:<22s}' + ''.join(f'{w[c]:>10.3f}' for c in [0, 3, 7, 11]))

for beta in [0.99, 0.999, 0.9999]:
    Wc, bc = train_softmax(Htr, ytr, class_w=cb_weights(train_counts, beta))
    report(f'③ CB Loss β={beta}', per_class_acc(Wc, bc))

# LDAM: Δ_c ∝ n_c^(-1/4)
def ldam_margin(counts, c_max=0.5):
    d = 1.0 / np.asarray(counts, float) ** 0.25
    return d / d.max() * c_max

delta = ldam_margin(train_counts, 0.5)
print('\nLDAM 间隔 Δ_c:', np.round(delta, 3))
Wl, bl = train_softmax(Htr, ytr, margin=delta)
report('④ LDAM (无 DRW)', per_class_acc(Wl, bl))
Wl2, bl2 = train_softmax(Htr, ytr, margin=delta, class_w=cb_weights(train_counts, 0.999))
report('④ LDAM + DRW', per_class_acc(Wl2, bl2))

assert RESULTS['③ CB Loss β=0.999'][TAIL].mean() > per0[TAIL].mean() + 0.15
assert RESULTS['④ LDAM (无 DRW)'][TAIL].mean() < RESULTS['④ LDAM + DRW'][TAIL].mean() - 0.1, \
    'LDAM 单用收益有限, 必须配 DRW —— 这正是原论文的结论'
print('\n⚠️  **LDAM 单独用几乎没涨**（%.3f → %.3f），配上重加权才有效（→ %.3f）。'
      % (per0[TAIL].mean(), RESULTS['④ LDAM (无 DRW)'][TAIL].mean(),
         RESULTS['④ LDAM + DRW'][TAIL].mean()))
print('   这正是 LDAM-DRW 论文的核心：**先按原分布学表示, 后期再平衡决策边界**。')

## 4 · 负梯度淹没：EQL 与 EQLv2 的机制

现在换成**检测器真正用的 sigmoid 多标签头**：每个类一个独立二分类，
再加进 4000 个**背景候选框**（它们对所有类都只贡献负梯度）。

关键量：类别 j 的**累积正负梯度比** `g_j = Σ|∇⁺| / Σ|∇⁻|`。
稀有类的 `g_j` 会低到头部类的几分之一 → `z_j` 被持续压低 → **score 上不去 → 被阈值滤掉**。

In [ ]:
N_BG = 4000
Xbg = rng.normal(scale=1.4, size=(N_BG, D))       # 背景候选框（不属于任何类）
Hall = np.vstack([Htr, feat(Xbg)])
Yall = np.zeros((len(Hall), C)); Yall[np.arange(len(ytr)), ytr] = 1.0
IS_FG = np.zeros(len(Hall), bool); IS_FG[:len(ytr)] = True
n_pos_cnt = Yall.sum(0); n_neg_cnt = len(Hall) - n_pos_cnt

print('每个类的**样本数**正负比（sigmoid 头下, 负样本 = 背景 + 所有其他类的正样本）:')
for c in [0, 3, 7, 11]:
    print(f'  类 {c:2d}  n={int(n_pos_cnt[c]):5d}   正:负 = 1 : {n_neg_cnt[c]/n_pos_cnt[c]:.0f}')

def sig(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def train_bce(mode='plain', steps=1500, lr=0.05, alpha=4.0, gamma=12.0, mu=0.8,
              lam=0.02, p_keep=0.0, seed=0):
    '''sigmoid 多标签头 + Adam。
       mode='plain' 普通 BCE / 'eql' EQL v1 / 'eqlv2' EQLv2'''
    W = np.zeros((H, C)); b = np.full(C, -3.0)
    mW = np.zeros_like(W); vW = np.zeros_like(W)
    mb = np.zeros_like(b); vb = np.zeros_like(b)
    acc_pos = np.full(C, 1e-8); acc_neg = np.full(C, 1e-8)
    rare = (train_counts / train_counts.sum()) < lam       # EQL v1 的稀有类判定
    g_gen = np.random.default_rng(seed)
    for it in range(steps):
        Praw = sig(Hall @ W + b)
        Graw = Praw - Yall                                  # BCE 对 logit 的梯度
        if mode == 'eqlv2':
            g = acc_pos / np.maximum(acc_neg, 1e-12)
            fg = 1.0 / (1.0 + np.exp(-gamma * (g - mu)))    # 把 g 映射到 [0,1]
            q, r = 1.0 + alpha * (1.0 - fg), fg             # 放大正梯度 / 衰减负梯度
            wgt = np.where(Yall > 0, q[None, :], r[None, :])
        elif mode == 'eql':
            # 屏蔽"来自其他**前景**类正样本"的负梯度（**保留背景负梯度**）
            mask = (Yall == 0) & IS_FG[:, None] & rare[None, :]
            keep = g_gen.random(Graw.shape) < p_keep        # β~Bernoulli: 只屏蔽一部分
            wgt = np.where(mask & ~keep, 0.0, 1.0)
        else:
            wgt = np.ones_like(Graw)
        G = Graw * wgt
        acc_pos += np.abs(G * (Yall > 0)).sum(0)
        acc_neg += np.abs(G * (Yall == 0)).sum(0)
        gW, gb = (Hall.T @ G) / len(Hall), G.mean(0)
        t_, b1, b2, eps = it + 1, 0.9, 0.999, 1e-8
        mW = b1 * mW + (1 - b1) * gW; vW = b2 * vW + (1 - b2) * gW ** 2
        mb = b1 * mb + (1 - b1) * gb; vb = b2 * vb + (1 - b2) * gb ** 2
        W -= lr * (mW / (1 - b1 ** t_)) / (np.sqrt(vW / (1 - b2 ** t_)) + eps)
        b -= lr * (mb / (1 - b1 ** t_)) / (np.sqrt(vb / (1 - b2 ** t_)) + eps)
    return W, b, acc_pos / np.maximum(acc_neg, 1e-12)

SCORE_THR = 0.3           # 检测器的 score 阈值
def recall_at(W, b, thr=SCORE_THR):
    S = sig(Hte @ W + b)
    return np.array([(S[yte == c, c] >= thr).mean() for c in range(C)])

Wp, bp, g_plain = train_bce('plain')
rec_p = recall_at(Wp, bp)
print(f'\n{"":<20s}{"head":>8s}{"mid":>8s}{"tail":>8s}{"macro":>9s}   g_head / g_tail')
def rep_bce(name, rec, g):
    print(f'{name:<20s}{rec[HEAD].mean():>8.3f}{rec[MID].mean():>8.3f}'
          f'{rec[TAIL].mean():>8.3f}{rec.mean():>9.3f}   {g[HEAD].mean():.3f} / {g[TAIL].mean():.3f}')
rep_bce('BCE 基线', rec_p, g_plain)
print('\n每类累积正负梯度比 g_j:', np.round(g_plain, 3))
assert g_plain[TAIL].mean() < g_plain[HEAD].mean() - 0.15, '尾部类的 g_j 必须明显更低'
assert rec_p[TAIL].mean() < 0.4 * rec_p[HEAD].mean(), '尾部召回应被压得很低'
print(f'\n⚠️  尾部类 g_j = {g_plain[TAIL].mean():.3f} vs 头部 {g_plain[HEAD].mean():.3f}'
      f' —— **稀有类是被负梯度压死的**, 不是特征学不好。')
print(f'    结果: score@{SCORE_THR} 的尾部召回只有 {rec_p[TAIL].mean():.3f}'
      f'（头部 {rec_p[HEAD].mean():.3f}）—— **框可能定位得很准, 但根本输不出来**。')

In [ ]:
# EQL v1（屏蔽来自更常见前景类的负梯度）与 EQLv2（在线梯度均衡）
We, be, g_eql = train_bce('eql', lam=0.02, p_keep=0.0)
Wv, bv, g_v2 = train_bce('eqlv2')
rec_e, rec_v = recall_at(We, be), recall_at(Wv, bv)

print(f'{"":<20s}{"head":>8s}{"mid":>8s}{"tail":>8s}{"macro":>9s}   g_head / g_tail')
rep_bce('BCE 基线', rec_p, g_plain)
rep_bce('EQL v1', rec_e, g_eql)
rep_bce('EQLv2', rec_v, g_v2)

RESULTS['⑦ sigmoid: BCE 基线'] = rec_p
RESULTS['⑦ sigmoid: EQL v1'] = rec_e
RESULTS['⑦ sigmoid: EQLv2'] = rec_v

assert rec_e[TAIL].mean() > rec_p[TAIL].mean(), 'EQL v1 应抬高尾部召回'
assert rec_v[TAIL].mean() > rec_e[TAIL].mean(), 'EQLv2 应优于 EQL v1'
assert rec_v[TAIL].mean() > 1.5 * rec_p[TAIL].mean(), 'EQLv2 对尾部的提升应当显著'
assert g_v2[TAIL].mean() > g_plain[TAIL].mean() + 0.15, 'EQLv2 应把尾部的 g_j 拉回来'
assert rec_v[HEAD].mean() >= rec_p[HEAD].mean() - 0.02, 'EQLv2 不应牺牲头部'

print(f'\n✅ EQLv2 把尾部的 g_j 从 {g_plain[TAIL].mean():.3f} 拉到 {g_v2[TAIL].mean():.3f}'
      f'（目标 1.0）, 尾部召回 {rec_p[TAIL].mean():.3f} → {rec_v[TAIL].mean():.3f}'
      f'，**头部没掉**。')
print('   EQLv2 是一个**负反馈控制器**: g_j 低 → 正梯度放大到 (1+α)、负梯度衰减到 ~0;')
print('   一旦均衡回来, 加权自动退回中性。**不需要类别频率先验** —— 对不断新增类别的')
print('   TSR 类别表（200~500 项, 随法规更新）是决定性的优点。')
print('⚠️  前提: 这套机制是为 **sigmoid 多标签头** 设计的。softmax 头上各类 logit 相互竞争,')
print('    稀有类不会被独立压到 -∞, 该用 BAGS / logit adjustment 而不是 EQL。')

## 5 · Logit adjustment：零训练成本的一行修正

推导：`p_bal(y|x) ∝ p_train(y|x)/π_c` → logit 空间里就是 **减去 `τ·log π_c`**。
`τ=1` 是理论最优（模型完美拟合的前提下），实际最优 τ 常落在 0.5–1.0。

In [ ]:
pi = train_counts / train_counts.sum()            # 训练集类别先验 π_c
print('π_c =', np.round(pi, 5))
print('-log π_c =', np.round(-np.log(pi), 2), '  ← 稀有类被抬得更多')

print(f'\n{"τ":>6s}{"head":>9s}{"mid":>9s}{"tail":>9s}{"macro":>10s}')
best_tau, best_macro = 0.0, -1
for tau in [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]:
    per = per_class_acc(W0, b0, adj=-tau * np.log(pi))
    if per.mean() > best_macro: best_tau, best_macro = tau, per.mean()
    mark = ''
    print(f'{tau:>6.2f}{per[HEAD].mean():>9.3f}{per[MID].mean():>9.3f}'
          f'{per[TAIL].mean():>9.3f}{per.mean():>10.3f}{mark}')
print(f'→ 最优 τ = {best_tau}  (macro {best_macro:.3f})')

per_la = per_class_acc(W0, b0, adj=-1.0 * np.log(pi))
report('⑤ logit adjustment τ=1', per_la)
assert np.allclose(per_class_acc(W0, b0, adj=-0.0 * np.log(pi)), per0), 'τ=0 必须退回原预测'
assert per_la[TAIL].mean() > per0[TAIL].mean() + 0.25, '推理时先验校正应大幅抬高尾部'
assert per_la[HEAD].mean() < per0[HEAD].mean(), '代价是头部下降 —— **工作点被移动了**'
print(f'\n⚠️  代价: 头部 {per0[HEAD].mean():.3f} → {per_la[HEAD].mean():.3f}。'
      '**调 logit = 移动 PR 工作点**, 原来的 score 阈值与 NMS 全部失效, 必须重新标定。')
print('    在 TSR 里尤其危险: 把稀有类分数抬上去 = "停车让行"误报增加 = 不必要的急刹。')
print('    正确做法: **按类别分别标定阈值, 安全关键类用更保守的 τ**。')

# 训练时版本: 把 τ·log π 加进训练的 logit（推理时什么都不做）
Wla, bla = train_softmax(Htr, ytr, prior_logit=1.0 * np.log(pi))
report('⑤ logit-adjusted loss', per_class_acc(Wla, bla))
print('\n✅ 两个版本理论上等价; 训练时版本通常更稳（模型能自己适应这个偏置），代价是要重训。')
print('   **工程顺序: 先用推理时版本白嫖一次探天花板, 再决定要不要花一次训练。**')

## 6 · 解耦训练：表示是好的，坏掉的是分类器

两个五分钟就能做的诊断：
1. **`‖w_c‖` 与 `log n_c` 的相关系数**——高（>0.8）说明分类器被频率带偏
2. **NCM（最近类均值）准确率 vs 模型分类头准确率**——NCM 明显更高 ⇒ 表示没坏

然后是三种阶段二方案：**τ-normalize（零成本）/ cRT（只重训分类头）/ LWS**。

In [ ]:
# ── 诊断 ①: 分类器权重范数 vs 样本数 ──────────────────────────
norms = np.linalg.norm(W0, axis=0)
corr = float(np.corrcoef(np.log(train_counts), norms)[0, 1])
print(f"{'类别':>6s}{'n_c':>8s}{'‖w_c‖':>10s}")
for c in range(C):
    print(f'{c:>6d}{train_counts[c]:>8d}{norms[c]:>10.3f}')
print(f'\n**corr(‖w_c‖, log n_c) = {corr:.3f}**  ← >0.75 就是"分类器被频率带偏"的典型指纹')
assert corr > 0.70, f'长尾训练下应当出现强相关, 实际 {corr:.3f}'

# ── 诊断 ②: NCM —— 完全不用分类器, 只用特征均值 ────────────────
mu_c = np.stack([Htr[ytr == c].mean(0) for c in range(C)])
mu_n = mu_c / np.linalg.norm(mu_c, axis=1, keepdims=True)
He_n = Hte / np.linalg.norm(Hte, axis=1, keepdims=True)
pred_ncm = (He_n @ mu_n.T).argmax(1)
per_ncm = np.array([(pred_ncm[yte == c] == c).mean() for c in range(C)])
report('⑥ NCM（只用表示）', per_ncm)
assert per_ncm[TAIL].mean() > per0[TAIL].mean() + 0.15, \
    'NCM 的尾部应显著优于模型自己的分类头 —— 这就是"表示没坏"的证据'
print(f'\n✅ **NCM 尾部 {per_ncm[TAIL].mean():.3f} vs 分类头 {per0[TAIL].mean():.3f}** ——')
print('   同一份特征, 不训练任何分类器就能做得更好 ⇒ **坏掉的是最后那一层**。')

# ── 阶段二 A: τ-normalized（零训练成本）───────────────────────
print(f'\n{"τ-norm τ":>10s}{"head":>9s}{"mid":>9s}{"tail":>9s}{"macro":>10s}')
for tau in [0.0, 0.3, 0.5, 0.7, 1.0]:
    Wn = W0 / (norms ** tau)
    per = per_class_acc(Wn, np.zeros(C))
    print(f'{tau:>10.2f}{per[HEAD].mean():>9.3f}{per[MID].mean():>9.3f}'
          f'{per[TAIL].mean():>9.3f}{per.mean():>10.3f}')
Wn7 = W0 / (norms ** 0.7)
report('⑥ τ-normalized τ=0.7', per_class_acc(Wn7, np.zeros(C)))
per_tau0 = per_class_acc(W0, np.zeros(C))
print(f'\n💡 注意 τ=0 那一行（macro {per_tau0.mean():.3f}）已经比基线（{per0.mean():.3f}）高很多 ——')
print('   因为它把 **bias b 丢掉了**, 而 bias 恰恰编码了类别先验。')
print('   "扔掉 bias" 本身就是一次粗糙的先验校正, 这与第 5 节是同一件事的两种做法。')
assert per_tau0.mean() > per0.mean() + 0.05, '丢掉 bias 本身就有先验校正效果'

# ── 阶段二 B: cRT —— 冻结表示, 用平衡采样重训分类器 ─────────────
bal_w = (1.0 / train_counts)[ytr]                 # 类别平衡采样 ≡ 权重 1/n_c
Wcrt, bcrt = train_softmax(Htr, ytr, sample_w=bal_w)
report('⑥ cRT (重训分类头)', per_class_acc(Wcrt, bcrt))

# 对照: 从头就用平衡采样训练（表示也被重采样"污染"）—— 这里表示层是固定的,
# 所以两者数值相同; 真实网络中 cRT 会更好, 因为 backbone 保住了原分布的多样性。
assert RESULTS['⑥ cRT (重训分类头)'][TAIL].mean() > per0[TAIL].mean() + 0.2
assert RESULTS['⑥ τ-normalized τ=0.7'][TAIL].mean() > per0[TAIL].mean() + 0.2
print('\n✅ 三种阶段二方案（τ-norm 零成本 / cRT 极低成本 / LWS）收益接近 ——')
print('   说明它们做的是**同一件事**: 把被频率带偏的决策边界拉回来。')
print('⚠️  真实网络里 cRT 还有一层额外好处: **表示用原分布学（多样性最大），**')
print('   **分类器用平衡数据学（决策边界正确）** —— 重采样从头训会损害前者。')

## 7 · 全方法对比：尾部类召回一览

把所有方法放在同一张表里。**注意 sigmoid 组（BCE / EQL / EQLv2）与 softmax 组
不可直接比较**——指标定义不同（前者是 score@0.3 的召回，后者是 argmax 准确率），
所以分两块看各自的 Δ。

In [ ]:
def table(keys, title, base_key):
    base = RESULTS[base_key]
    print(f'\n=== {title} ===')
    print(f'{"方法":<26s}{"head":>8s}{"mid":>8s}{"tail":>8s}{"macro":>9s}'
          f'{"Δtail":>9s}{"Δhead":>9s}')
    for k in keys:
        p = RESULTS[k]
        print(f'{k:<26s}{p[HEAD].mean():>8.3f}{p[MID].mean():>8.3f}{p[TAIL].mean():>8.3f}'
              f'{p.mean():>9.3f}{p[TAIL].mean()-base[TAIL].mean():>+9.3f}'
              f'{p[HEAD].mean()-base[HEAD].mean():>+9.3f}')

soft = [k for k in RESULTS if not k.startswith('⑦')]
table(soft, 'softmax 头（指标 = 平衡测试集上的 argmax 准确率）', '① baseline (plain CE)')
table([k for k in RESULTS if k.startswith('⑦')],
      'sigmoid 多标签头（指标 = score@0.3 的召回）', '⑦ sigmoid: BCE 基线')

# 方法不是可加的：叠加会过度校正
Wstack, bstack = train_softmax(Htr, ytr, sample_w=sw,
                               class_w=cb_weights(train_counts, 0.999))
per_stack = per_class_acc(Wstack, bstack, adj=-1.0 * np.log(pi))
best_single = max(RESULTS[k].mean() for k in soft if k != '⓪ oracle (平衡数据)')
print(f'\nRFS + CB Loss + logit adjustment **三个叠加**: '
      f'head {per_stack[HEAD].mean():.3f} tail {per_stack[TAIL].mean():.3f} '
      f'macro {per_stack.mean():.3f}')
print(f'单用最好的方法 macro = {best_single:.3f}')
assert per_stack.mean() < best_single + 0.02, \
    '三个叠加不应显著优于单用最好的 —— 它们本质上在做同一件事'
assert per_stack[HEAD].mean() < per0[HEAD].mean() - 0.1, '叠加会**过度校正**, 头部被压'
print('⚠️  **这些方法的收益不是可加的**: 它们都在把有效类别先验拉平, 叠加 = 过度校正')
print('    （尾部误检暴涨、头部被压）。**选一个, 把它的强度参数 (p/β/τ) 调好**,')
print('    而不是全开 —— 这个认识能在消融实验里省掉一半无效组合。')

## ✏️ 练习 1：真正的 repeat factor sampling（多标签版）

上面每张图只含一个类，太简单了。现在实现检测里真正的版本：

`rfs_image_factors(image_labels, n_images, t)`
- `image_labels`: `List[Set[int]]`，第 i 张图包含的类别集合
- 先算 **图像频率** `f_c = 含类别 c 的图像数 / n_images`
- `r_c = max(1, sqrt(t / f_c))`
- **`r_i = max_{c ∈ i} r_c`**（图内取最大，不是平均！空集的图 `r_i = 1.0`）
- 返回 `(r_c: np.ndarray[C], r_i: np.ndarray[n_images])`

`C` 由标签里出现过的最大类别 id + 1 决定。

In [ ]:
def rfs_image_factors(image_labels, n_images, t):
    # TODO: ① 统计每个类出现在多少张图里 -> f_c
    #       ② r_c = max(1, sqrt(t/f_c))；f_c = 0 的类记 r_c = 1.0（数据里没有它）
    #       ③ r_i = max_{c in i} r_c，空集图 r_i = 1.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
imgs = [{0}, {0}, {0}, {0, 1}, {0, 2}, {1}, {2, 3}, {0}, {0}, {0}]   # 10 张图, 4 个类
rc, ri = rfs_image_factors(imgs, len(imgs), t=0.5)
# f = [0.8, 0.2, 0.2, 0.1]
assert rc.shape == (4,) and ri.shape == (10,)
assert np.isclose(rc[0], 1.0), 'f_0=0.8 > t=0.5 -> 不重复'
assert np.isclose(rc[1], np.sqrt(0.5 / 0.2)), rc[1]
assert np.isclose(rc[3], np.sqrt(0.5 / 0.1)), rc[3]
assert np.isclose(ri[3], max(rc[0], rc[1])), '图内取 **最大**'
assert np.isclose(ri[6], max(rc[2], rc[3])) and np.isclose(ri[6], rc[3])
assert np.isclose(ri[0], 1.0)
# **共现污染**: 第 4 张图因为含稀有类 2 而被重复, 里面的常见类 0 也跟着被过采样
assert ri[4] > 1.0 and 0 in imgs[4]
# t 越大, 过采样越强
rc2, _ = rfs_image_factors(imgs, len(imgs), t=2.0)
assert (rc2 >= rc).all() and rc2[3] > rc[3]
print('f_c =', np.round(np.array([sum(c in s for s in imgs) for c in range(4)]) / 10, 2))
print('r_c =', np.round(rc, 3))
print('r_i =', np.round(ri, 3))
print('✅ 练习 1 通过：**图像频率 / 开方 / 图内取最大** —— 三个设计点缺一不可。')
print('   注意第 4 张图（含常见类0 + 稀有类2）被重复 %.2f 次 → **共现污染**。' % ri[4])

## ✏️ 练习 2：Class-Balanced 权重与"等效 β"

实现 `cb_weight_and_equiv(counts, beta, normalize=True)`，返回
`(w, p_equiv)`：
- `w`：CB 权重 `∝ (1-β)/(1-β^n_c)`；`normalize=True` 时把均值归一到 1
- `p_equiv`：把 CB 权重近似成幂律 `w_c ∝ n_c^(-p)` 时的等效指数 p
  （用 `head` 与 `tail` 两端做两点估计：`p = -(log w_tail - log w_head)/(log n_tail - log n_head)`，
  其中 head/tail 取 `counts` 的最大值与最小值对应的权重）

`p_equiv` 的意义：**p=0 等价不加权，p=1 等价反频率**——它把 β 翻译成人能直觉理解的强度。

In [ ]:
def cb_weight_and_equiv(counts, beta, normalize=True):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
w_lo, p_lo = cb_weight_and_equiv(train_counts, 0.9)
w_hi, p_hi = cb_weight_and_equiv(train_counts, 0.999999)
w_mid, p_mid = cb_weight_and_equiv(train_counts, 0.999)
assert np.isclose(w_lo.mean(), 1.0) and np.isclose(w_hi.mean(), 1.0)
assert p_lo < 0.15, f'β 很小时几乎等价不加权, p≈0, 实际 {p_lo:.3f}'
assert p_hi > 0.95, f'β→1 时等价反频率, p≈1, 实际 {p_hi:.3f}'
assert p_lo < p_mid < p_hi, '等效强度应随 β 单调递增'
assert (np.diff(w_mid) > 0).all(), 'counts 递减 -> 权重应递增'
w_raw, _ = cb_weight_and_equiv(train_counts, 0.999, normalize=False)
assert np.allclose(w_raw / w_raw.mean(), w_mid), 'normalize 只是整体缩放'
print(f"{'β':>12s}{'等效指数 p':>12s}   解读")
for beta in [0.0, 0.9, 0.99, 0.999, 0.9999, 0.999999]:
    _, p = cb_weight_and_equiv(train_counts, beta)
    tag = '≈不加权' if p < 0.15 else ('≈反频率' if p > 0.9 else '中间强度')
    print(f'{beta:>12.6f}{p:>12.3f}   {tag}')
print('✅ 练习 2 通过：**β 是"不加权 ↔ 反频率"之间的连续旋钮**, 用等效 p 来读它最直观。')

## ✏️ 练习 3：Logit adjustment（推理版 + 训练版）

实现 `logit_adjust(logits, priors, tau=1.0, mode='infer')`：
- `mode='infer'`：返回 `logits - tau*log(priors)`（推理时先验校正）
- `mode='train'`：返回 `logits + tau*log(priors)`（logit-adjusted loss 用的偏置版）
- `priors` 会被先归一化成概率（允许传入原始计数）
- `tau=0` 时两种模式都必须原样返回 logits

In [ ]:
def logit_adjust(logits, priors, tau=1.0, mode='infer'):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
z = Hte @ W0 + b0
assert np.allclose(logit_adjust(z, train_counts, tau=0.0), z), 'τ=0 必须原样返回'
assert np.allclose(logit_adjust(z, train_counts, tau=0.0, mode='train'), z)
a1 = logit_adjust(z, train_counts, tau=1.0)
a2 = logit_adjust(z, train_counts / train_counts.sum(), tau=1.0)
assert np.allclose(a1, a2), 'priors 应当被自动归一化（传计数或概率都行）'
assert np.allclose(logit_adjust(z, train_counts, 1.0, 'train'),
                   2 * z - logit_adjust(z, train_counts, 1.0, 'infer')), 'train/infer 反号'
# 校正后尾部类应当被抬起来
acc_before = np.array([(z.argmax(1)[yte == c] == c).mean() for c in range(C)])
acc_after = np.array([(a1.argmax(1)[yte == c] == c).mean() for c in range(C)])
assert acc_after[TAIL].mean() > acc_before[TAIL].mean() + 0.25
assert acc_after[HEAD].mean() < acc_before[HEAD].mean(), '代价是头部下降'
print(f"{'τ':>6s}{'head':>9s}{'tail':>9s}{'macro':>10s}")
for tau in [0.0, 0.5, 1.0, 1.5]:
    aa = logit_adjust(z, train_counts, tau=tau)
    pc = np.array([(aa.argmax(1)[yte == c] == c).mean() for c in range(C)])
    print(f'{tau:>6.1f}{pc[HEAD].mean():>9.3f}{pc[TAIL].mean():>9.3f}{pc.mean():>10.3f}')
print('✅ 练习 3 通过：**零训练成本、一行代码**, 应该是你遇到长尾时第一个试的东西。')

## ✏️ 练习 4：方法选择决策树

把讲解最后一节的决策树代码化。实现 `choose_method(diag)`，`diag` 含：
- `tail_recall`（尾部类的召回，框有没有出来）
- `tail_score_ok`（bool：尾部类的分数是否够高、能过阈值）
- `tail_confused_with_head`（bool：尾部主要被错分成相似的头部类）
- `tail_min_images`（尾部类最少的训练图像数）
- `w_norm_corr`（‖w_c‖ 与 log n_c 的相关系数）

返回 `{'diagnosis': str, 'actions': [str, ...]}`，`actions` **按成本递增排序**。
判定优先级（从上往下，命中即返回）：
1. `tail_min_images < 30` → `'信息量不足'`，actions = `['copy-paste 合成', '定向挖掘补数据']`
2. `tail_recall < 0.1` → `'检出问题'`，actions = `['检查标签分配', '检查漏标', '按小目标处理']`
3. `tail_confused_with_head` → `'特征不可分'`，actions = `['层次标签', '高分辨率 crop / 两级架构', '难例挖掘']`
4. `not tail_score_ok` 且 `w_norm_corr > 0.75` → `'分类器先验偏置'`，
   actions = `['logit adjustment', 'τ-normalize', 'cRT', 'RFS / EQLv2']`
5. 否则 → `'无明显长尾病征'`，actions = `['先检查评测口径（macro/分桶）']`

In [ ]:
def choose_method(diag):
    # TODO: 按 1→5 的优先级判定，命中即返回
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
base_diag = dict(tail_recall=0.55, tail_score_ok=False, tail_confused_with_head=False,
                 tail_min_images=800, w_norm_corr=0.87)
r = choose_method(base_diag)
assert r['diagnosis'] == '分类器先验偏置' and r['actions'][0] == 'logit adjustment', r
assert r['actions'][-1] == 'RFS / EQLv2', '成本递增: 零成本的在前'

d = dict(base_diag, tail_min_images=12)
assert choose_method(d)['diagnosis'] == '信息量不足', '样本太少时一切重加权都无效'
assert choose_method(d)['actions'] == ['copy-paste 合成', '定向挖掘补数据']

d = dict(base_diag, tail_recall=0.03)
assert choose_method(d)['diagnosis'] == '检出问题'
assert '检查标签分配' in choose_method(d)['actions']

d = dict(base_diag, tail_confused_with_head=True)
assert choose_method(d)['diagnosis'] == '特征不可分'

d = dict(base_diag, tail_score_ok=True, w_norm_corr=0.2)
assert choose_method(d)['diagnosis'] == '无明显长尾病征'

# 优先级：样本太少 **压过** 一切
d = dict(tail_recall=0.02, tail_score_ok=False, tail_confused_with_head=True,
         tail_min_images=8, w_norm_corr=0.95)
assert choose_method(d)['diagnosis'] == '信息量不足'

# 用本 notebook 真实测出来的数诊断一次
real = dict(tail_recall=float(rec_p[TAIL].mean()),
            tail_score_ok=bool(rec_p[TAIL].mean() > 0.5),
            tail_confused_with_head=False,
            tail_min_images=int(train_counts.min()),
            w_norm_corr=corr)
print('用本 notebook 的真实诊断量:', json.dumps(
    {k: (round(v, 3) if isinstance(v, float) else v) for k, v in real.items()},
    ensure_ascii=False))
print('→', json.dumps(choose_method(real), ensure_ascii=False, indent=2))
print('✅ 练习 4 通过：**先诊断再开药** —— 面试问"长尾怎么办"时能分层作答的人极少。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def rfs_image_factors(image_labels, n_images, t):
    C_ = max((max(s) for s in image_labels if s), default=-1) + 1
    img_cnt = np.zeros(C_, dtype=float)
    for s in image_labels:
        for c in s:
            img_cnt[c] += 1
    f_c = img_cnt / n_images
    with np.errstate(divide='ignore', invalid='ignore'):
        r_c = np.where(f_c > 0, np.maximum(1.0, np.sqrt(t / np.maximum(f_c, 1e-12))), 1.0)
    r_i = np.array([max((r_c[c] for c in s), default=1.0) for s in image_labels])
    return r_c, r_i

In [ ]:
# 练习 2 参考答案
def cb_weight_and_equiv(counts, beta, normalize=True):
    n = np.asarray(counts, float)
    En = np.ones_like(n) if beta <= 0 else (1.0 - beta ** n) / (1.0 - beta)
    w = 1.0 / En
    if normalize:
        w = w / w.mean()
    i_hi, i_lo = int(np.argmax(n)), int(np.argmin(n))       # 样本最多 / 最少的类
    dn = np.log(n[i_lo]) - np.log(n[i_hi])
    p = 0.0 if abs(dn) < 1e-12 else -(np.log(w[i_lo]) - np.log(w[i_hi])) / dn
    return w, float(p)

In [ ]:
# 练习 3 参考答案
def logit_adjust(logits, priors, tau=1.0, mode='infer'):
    p = np.asarray(priors, float)
    p = p / p.sum()
    delta = tau * np.log(p)
    return logits - delta if mode == 'infer' else logits + delta

In [ ]:
# 练习 4 参考答案
def choose_method(diag):
    if diag['tail_min_images'] < 30:
        return {'diagnosis': '信息量不足',
                'actions': ['copy-paste 合成', '定向挖掘补数据']}
    if diag['tail_recall'] < 0.1:
        return {'diagnosis': '检出问题',
                'actions': ['检查标签分配', '检查漏标', '按小目标处理']}
    if diag['tail_confused_with_head']:
        return {'diagnosis': '特征不可分',
                'actions': ['层次标签', '高分辨率 crop / 两级架构', '难例挖掘']}
    if (not diag['tail_score_ok']) and diag['w_norm_corr'] > 0.75:
        return {'diagnosis': '分类器先验偏置',
                'actions': ['logit adjustment', 'τ-normalize', 'cRT', 'RFS / EQLv2']}
    return {'diagnosis': '无明显长尾病征', 'actions': ['先检查评测口径（macro/分桶）']}

---
## 🧪 真实工程胶囊：长尾问题的十分钟诊断 + 按成本递增的处方

In [ ]:
RECIPE = r'''
# ============ 第 0 步：先把评测改对（不然优化的是错的东西）============
# 报告必须包含: macro AP + frequent/common/rare 三桶分项 + 逐类 AP
# LVIS 口径: rare < 10 imgs, common 10-100, frequent > 100（**用图像数, 不是实例数**）

# ============ 第 1 步：十分钟诊断（决定用哪一类方法）============
# ① 尾部类 recall 是不是也 ≈ 0？  -> 是: **检出问题**, 重加权救不了
#    去查: 标签分配（尾部框分到正样本了吗）/ anchor 尺度 / 是否同时是小目标 / 是否漏标严重
# ② ‖w_c‖ 与 log(n_c) 的相关系数
import torch, numpy as np
W = model.bbox_head.cls_convs[-1].weight.detach()      # 或 roi_head.bbox_head.fc_cls.weight
norms = W.flatten(1).norm(dim=1).cpu().numpy()
corr  = np.corrcoef(np.log(class_counts), norms)[0, 1]
print("corr(||w_c||, log n_c) =", corr)                # > 0.8 -> **分类器被频率带偏**
# ③ NCM 诊断: 冻结 backbone, 用每类特征均值做最近邻分类
#    NCM 尾部准确率 >> 模型分类头 -> **表示没坏, 坏的是最后一层**

# ============ 第 2 步：按成本递增开药 ============
# --- 零成本（半小时, 不重训）---
tau = 1.0
logits_adj = logits - tau * torch.log(torch.tensor(class_priors))     # logit adjustment
W_norm = W / W.flatten(1).norm(dim=1).view(-1,1,1,1) ** 0.7           # τ-normalize
# ⚠️ 分数分布整体平移 -> **score 阈值与 NMS 必须重新标定**, 按类别分别定阈值

# --- 低成本（几分钟, 只重训分类头）---
# cRT: 冻结 backbone + neck, 只用 class-balanced sampling 重训分类分支
for p in model.backbone.parameters(): p.requires_grad = False
for p in model.neck.parameters():     p.requires_grad = False

# --- 改训练（一次完整训练）---
# mmdet: repeat factor sampling
dataset = dict(type='ClassBalancedDataset', oversample_thr=1e-3, dataset=dict(...))
#   ⚠️ epoch 变长 1.5~2x -> **必须同步调 max_iters 与 lr schedule**, 否则消融不公平
# EQLv2（sigmoid 头必选; 无需类别频率先验, 对不断新增类别友好）
loss_cls = dict(type='EQLV2Loss', num_classes=NUM_CLASSES, gamma=12, mu=0.8, alpha=4.0)

# --- 最贵但唯一提高信息量上限 ---
# copy-paste 稀有类实例（C56 m03）+ 定向挖掘补数据（C58 m03-m05）
# 架构: 两级方案（类别无关检测 -> 高分辨率 crop 分类），把长尾搬出检测器
#   代价: **级联召回 = 检测召回 x 分类准确率**（0.95 x 0.95 = 0.90）

# ============ 红线 ============
# 1. **Focal Loss 不是长尾解法** —— 它按难度加权, 不看类别频率
# 2. **这些方法收益不可加** —— RFS + CB + logit adj 叠加 = 过度校正, 选一个调好
# 3. **RFS 改了 epoch 长度** —— 不同步调 iteration 数的消融全部无效
# 4. TSR: 抬高稀有类分数 = "停车让行"误报增加 = 急刹。**安全关键类用更保守的 τ**
'''
print(RECIPE)
for k in ['corr(||w_c||', 'NCM', 'logit adjustment', 'τ-normalize', 'cRT',
          'ClassBalancedDataset', 'EQLV2Loss', '级联召回', 'Focal Loss 不是长尾解法']:
    assert k in RECIPE, k
print('✅ 胶囊覆盖: 评测口径 → 十分钟诊断 → 按成本递增的四档处方 → 四条红线。')

### 小结

- **检测里有两种不平衡，解法互不替代**：前景-背景（位置级，每张图都有，1:280~1:2800，
  用 Focal/OHEM/分配）vs 类别间（数据集级，用 RFS/CB/EQL/logit adj/解耦）。
  **「长尾怎么办」答「用 Focal Loss」是把两个问题混为一谈。**
- **RFS 的四个设计点**：图像频率 `f_c`、开方（温和过采样，不是完全拉平）、
  图内取最大、每 epoch 随机取整。副作用：共现污染 + epoch 长度变化（消融不公平的常见来源）。
- **CB Loss 的 β 是连续旋钮**：`E_n=(1-β^n)/(1-β)`，β→0 等价不加权、β→1 等价反频率，
  `1/(1-β)` 就是「多少样本算饱和」。**LDAM 单用几乎没涨，必须配 DRW。**
- **稀有类是被负梯度淹死的**：sigmoid 头下尾部类累积正负梯度比远低于头部，
  logit 被压 → score 上不去 → 框定位准也输不出来。**EQL 屏蔽、EQLv2 在线均衡到 1:1**，
  后者不需要频率先验，对类别表持续增长的 TSR 特别合适。
- **表示是好的，坏掉的是分类器**：`corr(‖w_c‖, log n_c) > 0.8` + `NCM 尾部远优于分类头`
  是两个五分钟诊断。对应处方：τ-normalize（零成本）、cRT（只重训分类头）、LWS。
- **logit adjustment 是第一个该试的**：推理时减 `τ·log π_c`，零训练成本。
  代价是**工作点被移动**——score 阈值与 NMS 必须重新标定，安全关键类要更保守。
- **这些方法收益不可加**：它们都在拉平有效类别先验，叠加 = 过度校正。**选一个，调好强度。**
- **先诊断再开药**：样本太少（<30 张）→ 只能补数据；recall≈0 → 检出问题；
  被错分成头部类 → 特征不可分；score 低 + 权重范数相关 → 分类器先验偏置。

下一站：**模块 02 · 难例挖掘** —— 在已有数据里找出该重点学的那一小撮，
以及为什么「挖掘过头会把噪声标签当难例，反而毁掉模型」。